# Support plots for BaLLMatro docs

## Computational complexity VS number of cards

In [ ]:
import math
import plotly.express as px

# Complexity: since the smallest poker hand contains 0 cards and the largest poker hand contains 5 cards, the complexity of finding the optimal hand among n cards is \sum_{i=0}^{n} C(n, i)
cards = list(range(0, 21))
complexities = [sum(math.comb(n, i) for i in range(6)) for n in cards]

fig = px.line(x=cards, y=complexities, labels={'x': 'Number of Cards', 'y': 'Complexity'}, text=complexities)
fig.update_traces(line=dict(width=4), textposition='top center', marker=dict(size=8))

fig.update_layout(font=dict(size=20), margin=dict(l=10, r=10, t=10, b=10), yaxis_range=[0, 25000])
fig.show()
# Plotly figures do not have a savefig method. Use write_image instead.
fig.write_image('complexity.png', scale=3, width=1800, height=600)

## Differences in scores when removing jokers from dataset

In [ ]:
from datasets import load_dataset
from ballmatro.card import parse_card_list
from ballmatro.optimizer import brute_force_optimize
from ballmatro.score import  ScoreDataset

scores = {}
for i in ["5", "6", "7", "X"]:
    level = f"level{i}"
    ds = load_dataset("albarji/ballmatro", level)
    original_score = sum(row["score"] for row in ds["test"])

    dejokered_plays = []
    for row in ds["test"]:
        dejokered_cards = [c for c in parse_card_list(row["input"]) if not c.is_joker]
        optimal_dejokered_play = brute_force_optimize(dejokered_cards).played
        dejokered_plays.append(optimal_dejokered_play)

    dejokered_score = ScoreDataset(ds["test"], dejokered_plays).total_normalized_score

    scores[level] = {"original": 1.0, "dejokered": dejokered_score, "joker_effect": 1 - dejokered_score}

scores